# Overmerge Cluster Analysis

Compare three clustering approaches on labeled gold standard authors
(overmerged vs clean) to find the best overmerge detection signal.

- Approach 2: K=2 silhouette on work embeddings
- Approach 3: Pairwise work-to-work cosine similarity
- Job: #105.11 aer-overmerge-signal-calibration

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity as cos_sim
import time
from datetime import datetime

WORK_EMBEDDINGS_TABLE = "openalex.vector_search.work_embeddings_v2"
WORKS_TABLE = "openalex.works.openalex_works"
AUTHOR_EMBEDDINGS_TABLE = "openalex.aer.author_embeddings"
EMBEDDING_DIM = 1024

## Define labeled authors

In [ ]:
# Zendesk overmerge cases
zendesk_overmerged = [
    5106038276, 5127355048, 5121306932, 5071424520, 5088402891,
    5045317380, 5100726594, 5108072493, 5009709588, 5045904720,
    5012309101, 5069151214, 5011165427
]

# Gold standard splits (overmerged)
gold_splits = [
    5085619921, 5041864386, 5117075932, 5110196385, 5065096990,
    5037835036, 5071539405, 5017142455, 5110027945, 5091859986,
    5108623528, 5101092520, 5070168570, 5072305232, 5013478230,
    5009307852, 5016882470, 5091442005, 5079979698, 5108570566,
    5110358878, 5079166689, 5108748587, 5102755488, 5079652777,
    5113385911, 5113952815, 5010422511, 5007114677, 5037924045,
    5025452752, 5088152507, 5073218194, 5112257425, 5061138371,
    5033771456, 5101839347, 5074059264, 5080312909, 5010975613,
    5048151405, 5052571385, 5111372795, 5112390495, 5103459170,
    5103347958, 5062125091, 5110749239, 5021173676, 5113857253,
    5108453412, 5076836352, 5060512575, 5079413276, 5079617160,
    5055163904, 5064679635, 5109293548, 5067290858, 5050939922,
    5056229359, 5037435179, 5006626689, 5103976114, 5010484703,
    5087117526, 5019217807, 5075755572, 5109148503, 5014188192,
    5051189473, 5058421597, 5057408903, 5056945568, 5074432852,
    5019862321, 5103907138, 5003683328, 5103173829, 5051468011,
    5038742031, 5062220638, 5114098473, 5001106162, 5029904482,
    5073480387, 5002981530
]

# Gold standard confirmed (clean)
gold_confirmed = [
    5087166684, 5081747147, 5082325782, 5012168214, 5012434282,
    5003686344, 5010617958, 5084195402, 5079654941, 5000848656,
    5061377970, 5000447750, 5023844756, 5080591937, 5027129242,
    5067354901, 5109968665, 5037064192, 5072647734, 5016361065,
    5026928687, 5002615359, 5027008194, 5063093897, 5108260432,
    5011946452, 5079497159, 5083508133, 5109604058, 5079798553,
    5088931673, 5060557374, 5086191294, 5044638100, 5036926833,
    5037676122, 5082406391, 5073380969, 5019534047, 5090821911,
    5111962545, 5027442102, 5036629892, 5084506603, 5075560544,
    5114030225, 5047130741, 5088150711, 5024736221, 5046646750,
    5111945307, 5066075206, 5063684507, 5053820618, 5055178909,
    5087090962, 5103243948, 5050549195, 5033358050, 5102850112,
    5055945041, 5026328537, 5022112939, 5011576899, 5036961392,
    5000185973, 5013710631, 5088159296, 5063641432, 5018449545,
    5074749021, 5052625144, 5078623160, 5030659236, 5004860476,
    5002085848, 5088382884, 5050861768, 5070669440, 5035638193,
    5027340715, 5067395854, 5018332726, 5000163809, 5048092761,
    5063197873, 5077040304, 5033702363, 5110246466, 5074731779,
    5015418171, 5052638321, 5085326237, 5061312215, 5075570397,
    5112171421, 5050680576, 5017797591, 5065396267, 5087110857
]

overmerged_set = set(zendesk_overmerged + gold_splits)
clean_set = set(gold_confirmed)
all_authors = list(overmerged_set | clean_set)

print(f"Overmerged: {len(overmerged_set)}, Clean: {len(clean_set)}, Total: {len(all_authors)}")

## Pull work embeddings for labeled authors

In [ ]:
# Pull all work embeddings for labeled authors (small enough to collect)
author_ids_str = ','.join(str(a) for a in all_authors)

df = spark.sql(f"""
    SELECT
        CAST(REPLACE(a.author.id, 'https://openalex.org/A', '') AS BIGINT) AS author_id,
        e.work_id,
        e.embedding
    FROM {WORK_EMBEDDINGS_TABLE} e
    JOIN {WORKS_TABLE} w ON e.work_id = CAST(w.id AS STRING)
    LATERAL VIEW OUTER EXPLODE(w.authorships) AS a
    WHERE a.author.id IS NOT NULL
      AND e.embedding IS NOT NULL
      AND CAST(REPLACE(a.author.id, 'https://openalex.org/A', '') AS BIGINT) IN ({author_ids_str})
""")

# Collect to driver — small enough for ~500 authors
rows = df.collect()
print(f"Collected {len(rows):,} (author, work, embedding) rows")

# Group by author
from collections import defaultdict
author_works = defaultdict(list)
for row in rows:
    emb = np.array(row['embedding'][:EMBEDDING_DIM], dtype=np.float64)
    if not np.any(np.isnan(emb)):
        author_works[row['author_id']].append({
            'work_id': row['work_id'],
            'embedding': emb
        })

print(f"Authors with embeddings: {len(author_works)}")
print(f"Median works per author: {np.median([len(v) for v in author_works.values()]):.0f}")

## Approach 2: K=2 Silhouette Score

In [ ]:
def compute_silhouette_k2(embeddings):
    """Run k-means k=2 on embeddings, return silhouette score."""
    if len(embeddings) < 4:  # need at least 2 per cluster
        return None
    X = np.array(embeddings)
    km = KMeans(n_clusters=2, n_init=10, random_state=42)
    labels = km.fit_predict(X)
    # Check both clusters have members
    if len(set(labels)) < 2:
        return None
    return silhouette_score(X, labels, metric='cosine')

results = []
start = time.time()
for author_id, works in author_works.items():
    if len(works) < 5:
        continue
    embeddings = [w['embedding'] for w in works]
    sil = compute_silhouette_k2(embeddings)
    label = 'overmerged' if author_id in overmerged_set else 'clean'
    results.append({
        'author_id': author_id,
        'label': label,
        'work_count': len(works),
        'silhouette_k2': sil
    })

elapsed = time.time() - start
print(f"Computed silhouette for {len(results)} authors in {elapsed:.1f}s")

# Summary by label
for label in ['overmerged', 'clean']:
    vals = [r['silhouette_k2'] for r in results if r['label'] == label and r['silhouette_k2'] is not None]
    print(f"\n{label} (n={len(vals)}):")
    print(f"  Mean silhouette: {np.mean(vals):.4f}")
    print(f"  Median:          {np.median(vals):.4f}")
    print(f"  Q1:              {np.percentile(vals, 25):.4f}")
    print(f"  Q3:              {np.percentile(vals, 75):.4f}")

## Approach 3: Pairwise Work-to-Work Cosine Similarity

In [ ]:
def compute_pairwise_stats(embeddings):
    """Compute pairwise cosine similarity stats for a set of embeddings."""
    if len(embeddings) < 2:
        return None
    X = np.array(embeddings)
    sim_matrix = cos_sim(X)
    # Extract upper triangle (exclude diagonal)
    n = len(X)
    upper_tri = sim_matrix[np.triu_indices(n, k=1)]
    return {
        'mean_pairwise': float(np.mean(upper_tri)),
        'min_pairwise': float(np.min(upper_tri)),
        'p10_pairwise': float(np.percentile(upper_tri, 10)),
        'std_pairwise': float(np.std(upper_tri)),
    }

pairwise_results = []
start = time.time()
for author_id, works in author_works.items():
    if len(works) < 5:
        continue
    embeddings = [w['embedding'] for w in works]
    stats = compute_pairwise_stats(embeddings)
    if stats is None:
        continue
    label = 'overmerged' if author_id in overmerged_set else 'clean'
    pairwise_results.append({
        'author_id': author_id,
        'label': label,
        'work_count': len(works),
        **stats
    })

elapsed = time.time() - start
print(f"Computed pairwise stats for {len(pairwise_results)} authors in {elapsed:.1f}s")

# Summary by label
for label in ['overmerged', 'clean']:
    subset = [r for r in pairwise_results if r['label'] == label]
    print(f"\n{label} (n={len(subset)}):")
    for metric in ['mean_pairwise', 'min_pairwise', 'p10_pairwise', 'std_pairwise']:
        vals = [r[metric] for r in subset]
        print(f"  {metric}: mean={np.mean(vals):.4f}, median={np.median(vals):.4f}, q1={np.percentile(vals, 25):.4f}, q3={np.percentile(vals, 75):.4f}")

## Combined comparison

In [ ]:
# Merge results and show the best-separating metrics
combined = {}
for r in results:
    if r['silhouette_k2'] is not None:
        combined[r['author_id']] = {'label': r['label'], 'silhouette_k2': r['silhouette_k2']}

for r in pairwise_results:
    if r['author_id'] in combined:
        combined[r['author_id']].update({
            'mean_pairwise': r['mean_pairwise'],
            'min_pairwise': r['min_pairwise'],
            'p10_pairwise': r['p10_pairwise'],
            'std_pairwise': r['std_pairwise'],
        })

# For each metric, compute the gap between overmerged and clean medians
print("Metric separation (median overmerged vs median clean):")
print("=" * 70)
metrics = ['silhouette_k2', 'mean_pairwise', 'min_pairwise', 'p10_pairwise', 'std_pairwise']
for metric in metrics:
    over_vals = [v[metric] for v in combined.values() if v['label'] == 'overmerged' and metric in v]
    clean_vals = [v[metric] for v in combined.values() if v['label'] == 'clean' and metric in v]
    if over_vals and clean_vals:
        over_med = np.median(over_vals)
        clean_med = np.median(clean_vals)
        gap = abs(over_med - clean_med)
        # Effect size: gap relative to pooled std
        pooled_std = np.std(over_vals + clean_vals)
        effect_size = gap / pooled_std if pooled_std > 0 else 0
        direction = '↑' if over_med > clean_med else '↓'
        print(f"  {metric:20s}: overmerged={over_med:.4f} clean={clean_med:.4f} gap={gap:.4f} effect={effect_size:.2f} {direction}")